In [2]:
import pandas as pd

In [4]:
df =  pd.read_csv("dataset.csv")
df.head()

,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,num_failed_logins,logged_in,num_compromised,...,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,duration_label,protocol_type_label,service_label,dst_host_srv_rerror_rate_label
0,491,0,0,0,0,0,0,0,0,0,...,0.17,0.00,0.00,0.00,0.05,0.00,1,20,9,16
1,146,0,0,0,0,0,0,0,0,0,...,0.88,0.00,0.00,0.00,0.00,0.00,2,44,9,16
2,0,0,0,0,0,0,0,0,0,0,...,0.00,0.00,1.00,1.00,0.00,0.00,1,49,5,14
3,232,8153,0,0,0,0,0,1,0,0,...,0.03,0.04,0.03,0.01,0.00,0.01,1,24,9,16
4,199,420,0,0,0,0,0,1,0,0,...,0.00,0.00,0.00,0.00,0.00,0.00,1,24,9,16


In [5]:
df. isnull().sum()

flag                              0
src_bytes                         0
dst_bytes                         0
land                              0
wrong_fragment                    0
urgent                            0
hot                               0
num_failed_logins                 0
logged_in                         0
num_compromised                   0
root_shell                        0
su_attempted                      0
num_root                          0
num_file_creations                0
num_shells                        0
num_access_files                  0
num_outbound_cmds                 0
is_host_login                     0
is_guest_login                    0
count                             0
srv_count                         0
serror_rate                       0
srv_serror_rate                   0
rerror_rate                       0
srv_rerror_rate                   0
same_srv_rate                     0
diff_srv_rate                     0
srv_diff_host_rate          

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 148517 entries, 0 to 148516
Data columns (total 41 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   flag                            148517 non-null  int64  
 1   src_bytes                       148517 non-null  int64  
 2   dst_bytes                       148517 non-null  int64  
 3   land                            148517 non-null  int64  
 4   wrong_fragment                  148517 non-null  int64  
 5   urgent                          148517 non-null  int64  
 6   hot                             148517 non-null  int64  
 7   num_failed_logins               148517 non-null  int64  
 8   logged_in                       148517 non-null  int64  
 9   num_compromised                 148517 non-null  int64  
 10  root_shell                      148517 non-null  int64  
 11  su_attempted                    148517 non-null  int64  
 12  num_root        

In [ ]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch_geometric.data import Data
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from torch_geometric.loader import DataLoader
import numpy as np

In [ ]:
# Define the GCN model
class GCNModel(torch.nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, dropout_rate=0.5):
        super(GCNModel, self).__init__()
        self.conv1 = GCNConv(input_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.fc = torch.nn.Linear(hidden_dim, output_dim)
        self.dropout_rate = dropout_rate

    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout_rate, training=self.training)
        x = F.relu(self.conv2(x, edge_index))
        x = F.dropout(x, p=self.dropout_rate, training=self.training)
        x = self.fc(x)
        return F.log_softmax(x, dim=1)

In [ ]:
# Data preparation
features = df.drop('dst_host_srv_rerror_rate_label', axis=1).values
target = df['dst_host_srv_rerror_rate_label'].values

scaler = StandardScaler()
features = scaler.fit_transform(features)

In [ ]:
# Splitting data
X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.2, random_state=42, stratify=target)

In [ ]:
# Convert to torch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float)
X_test_tensor = torch.tensor(X_test, dtype=torch.float)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

In [ ]:
# Define edge index (must be provided based on graph structure)
edge_index = torch.tensor([[0, 1], [1, 2]], dtype=torch.long)  # Placeholder for actual graph structure

data = Data(x=X_train_tensor, edge_index=edge_index, y=y_train_tensor)
data_list = [data]
data_loader = DataLoader(data_list, batch_size=32, shuffle=True)

In [ ]:
# Model initialization
input_dim = X_train.shape[1]
hidden_dim = 64
output_dim = len(np.unique(target))
dropout_rate = 0.5
model = GCNModel(input_dim, hidden_dim, output_dim, dropout_rate)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

In [ ]:
# Training the model
model.train()
for epoch in range(100):
    total_loss = 0
    for batch in data_loader:
        optimizer.zero_grad()
        out = model(batch.x, batch.edge_index)
        loss = F.nll_loss(out, batch.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f'Epoch {epoch+1}, Loss: {total_loss:.4f}')

e:\Softwares\anaconda3\envs\kaggle\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Epoch 1, Loss: 3.6280
Epoch 2, Loss: 3.0667
Epoch 3, Loss: 2.4848
Epoch 4, Loss: 1.9058
Epoch 5, Loss: 1.4608
Epoch 6, Loss: 1.1845
Epoch 7, Loss: 1.0725
Epoch 8, Loss: 1.0277
Epoch 9, Loss: 0.9862
Epoch 10, Loss: 0.9318
Epoch 11, Loss: 0.8704
Epoch 12, Loss: 0.8217
Epoch 13, Loss: 0.7807
Epoch 14, Loss: 0.7421
Epoch 15, Loss: 0.7127
Epoch 16, Loss: 0.6820
Epoch 17, Loss: 0.6591
Epoch 18, Loss: 0.6316
Epoch 19, Loss: 0.6085
Epoch 20, Loss: 0.5856
Epoch 21, Loss: 0.5624
Epoch 22, Loss: 0.5459
Epoch 23, Loss: 0.5250
Epoch 24, Loss: 0.5116
Epoch 25, Loss: 0.5000
Epoch 26, Loss: 0.4883
Epoch 27, Loss: 0.4772
Epoch 28, Loss: 0.4665
Epoch 29, Loss: 0.4587
Epoch 30, Loss: 0.4493
Epoch 31, Loss: 0.4409
Epoch 32, Loss: 0.4344
Epoch 33, Loss: 0.4231
Epoch 34, Loss: 0.4168
Epoch 35, Loss: 0.4088
Epoch 36, Loss: 0.4025
Epoch 37, Loss: 0.3982
Epoch 38, Loss: 0.3928
Epoch 39, Loss: 0.3869
Epoch 40, Loss: 0.3827
Epoch 41, Loss: 0.3796
Epoch 42, Loss: 0.3705
Epoch 43, Loss: 0.3670
Epoch 44, Loss: 0.36

In [ ]:
# Evaluate the model
model.eval()
out = model(X_test_tensor, edge_index)
pred = out.argmax(dim=1).detach().numpy()
accuracy = accuracy_score(y_test, pred)
print(f'Accuracy: {accuracy:.4f}')

In [1]:
# Save the model
torch.save(model.state_dict(), 'gcn_model.pth')
print('Model saved as gcn_model.pth')

NameError: name 'torch' is not defined